In [1]:
from pathlib import Path
import numpy as np
import jax.numpy as jnp
from jax import random

from tensorflow_probability.substrates.jax import distributions as tfd
import optax

from sbijax import NPE
from sbijax.nn import make_maf

AttributeError: module 'jax.interpreters.xla' has no attribute 'pytype_aval_mappings'

### Import and prepare data

In [101]:
data_dir = Path("./dt-data-time1/")
# n_batches = 10
n_batches = 1 # smaller test case

theta_indices = (3,4,5,6,7,8) # only positions

In [102]:
theta_list = []
for i in range(n_batches):
    file_path = data_dir / f"theta_batch_{i:02d}.npy"
    theta = np.load(file_path)
    theta_list.append(theta)

theta = jnp.concatenate(theta_list, axis=0)[:, theta_indices]

In [103]:
x_list = []
for i in range(n_batches):
    file_path = data_dir / f"x_batch_{i:02d}.npy"
    x = np.load(file_path)       # shape: (N_batch, n_observables)
    x_list.append(x)

x = jnp.concatenate(x_list, axis=0)

mass_info = theta[:, :3]
x = jnp.concatenate([mass_info, x], axis=1)

In [104]:
print("Theta shape: ", theta.shape)
print("x shape:     ", x.shape)

Theta shape:  (9999, 6)
x shape:      (9999, 15)


### Set prior distribution

In [105]:
n_dim_theta = theta.shape[1]
n_dim_x = x.shape[1]

# Set prior
def prior_fn():
    low  = jnp.array([0.1, 0.1, 0.1, 0.1, 0, 0.1, 0, 0.1, 0, 0.1, 0, 0.1, 0, 0.1, 0])[np.array(theta_indices)]
    high =  jnp.array([10.0, 10.0, 10.0, 5.0, 10.0, 5.0, 10.0, 5.0, 10.0, 5.0, 10.0, 5.0, 10.0, 5.0, 10.0])[np.array(theta_indices)]
    prior = tfd.JointDistributionNamed(dict(
        theta=tfd.Independent(
            tfd.Uniform(low=low, high=high),
            reinterpreted_batch_ndims=1
        )
    ), batch_ndims=0)
    return prior

### Train NPE Model

In [106]:
# Define NPE model
neural_network = make_maf(n_dimension=n_dim_theta, n_layers=2)
fns = (prior_fn, None)
estim = NPE(fns, neural_network)

In [107]:
# Train model
data = {"theta": theta, "y": x}

params, info = estim.fit(
    random.PRNGKey(0),
    data=data,
    optimizer=optax.adam(1e-3),
    # n_early_stopping_patience=10,
    # n_early_stopping_delta=0.01,
)

print("Training finished.")

 10%|▉         | 98/1000 [01:51<17:06,  1.14s/it]

Training finished.


### Test the trained model with a seen case

In [109]:
x_obs = x[0]
posterior_samples, _ = estim.sample_posterior(
    random.PRNGKey(1),
    params,
    observable=x_obs,
    n_samples=1000,
)

In [120]:
print("True positions:")
print(theta[0])
print("\n")
print("NPE Prediction:")
print(jnp.mean(posterior_samples.posterior["theta"].values[0], axis=0))

True positions:
[2.7710838 7.1797094 1.1384773 5.11331   4.658235  8.597766 ]


NPE Prediction:
[2.7732327 7.194672  1.1468079 5.534941  4.4564524 8.813274 ]
